# Regression Track — Predicting Student Performance

**23CSE301 Machine Learning Capstone · Review 1**
**Project:** *Predictive Analytics Across Domains: Student Performance, Sentiment Classification and Customer Segmentation*

| | |
|---|---|
| **Problem** | Predict a student's `Performance Index` (a continuous score from 10 to 100) from study habits and prior results. |
| **Task type** | Supervised learning, **regression** (numeric target). |
| **Dataset** | *Student Performance (Multiple Linear Regression)*, Kaggle — 10,000 rows × 6 columns. |
| **Track owner** | _(student name)_ |

> ⚠️ **Synthetic data disclosure.** The dataset author states on the Kaggle page that the data is
> **synthetic and created for illustrative purposes**, and that the relationships may not reflect the
> real world. Every result in this notebook describes this synthetic data, not real students.

### Review 1 scope
This notebook covers the **complete regression track**: all ten algorithms required by guideline 3.1,
evaluated on the same held-out test set. Classification lives in `classification.ipynb`.
Clustering (and Classification Part B) belong to Review 2 and are deliberately not in this repository.

### How to read this notebook
Every section states **what** is done, **why**, and the **ML concept** behind it. Reusable plumbing
(paths, contracts, metric conventions, plot style) lives in `src/`; the machine-learning steps
themselves — building pipelines, training, tuning, evaluating — are written out in the cells below.
Cells marked **✍️ TEAM ANALYSIS REQUIRED** are for the team's own observations and must not be
written by an AI tool.

## 1. Set-up

**What:** import libraries, locate the project root, apply the plotting style and print the environment.

**Why:** reproducibility. Printing library versions and the random seed means anyone re-running the
notebook can confirm they are using the same environment that produced these results
(`requirements.txt` pins the exact versions).

**Concept — reproducibility:** many ML steps involve randomness (the train/test split, bootstrap
samples in Random Forest, tie-breaking). Fixing `random_state=42` everywhere makes every run produce
identical numbers.

In [1]:
# --- Environment set-up --------------------------------------------------------
import sys
import platform
from pathlib import Path

# Find the project root (the folder containing src/config.py). This works whether
# the notebook is opened from notebooks/ or from the project root, and avoids any
# hard-coded absolute path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from src import config, show_source
from src.plotting import set_style

set_style()                                   # titles, labels, colourblind palette
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Only folder NAMES are printed, so no personal absolute paths end up in the committed notebook.
print(f"Python        {platform.python_version()}  (environment: {Path(sys.prefix).name})")
print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} "
      f"| matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")
print(f"Project root  .../{ROOT.name}")
print(f"Random seed   {config.RANDOM_STATE}   test size {config.TEST_SIZE}   CV folds {config.CV_FOLDS}")
print(f"Run mode      {config.RUN_MODE}" + ("   (DEV RUN - outputs carry the '_dev' suffix, NOT reported results)"
                                             if config.RUN_MODE == "dev" else ""))

Python        3.13.2  (environment: ml-capstone)
numpy 2.5.3 | pandas 3.0.6 | scikit-learn 1.9.1 | matplotlib 3.11.2 | seaborn 0.13.2
Project root  .../Machine Learning project
Random seed   42   test size 0.2   CV folds 5
Run mode      full


In [2]:
# --- Is the raw data present and unchanged? ------------------------------------
# load_raw() verifies the SHA-256 checksum, row count and exact column list against
# the dataset contract in src/config.py and raises ContractError on any mismatch.
from src.data_loading import load_raw

contract = config.STUDENT
_df_check = load_raw(contract)
print(f"{contract.filename}: {_df_check.shape[0]:,} rows x {_df_check.shape[1]} columns - checksum and schema match the contract")
del _df_check

Student_Performance.csv: 10,000 rows x 6 columns - checksum and schema match the contract


## 2. Data loading and audit  *(rubric A1)*

**What:** load the raw CSV and report its shape, column types, missing values, unique counts and the
distribution of the target.

**Why:** before modelling we must know exactly what the data contains. A shape or type that differs
from what we expect is caught here instead of silently corrupting results later.

**Concept — data audit:** a systematic check of every column: its type (numeric/categorical), how many
values are missing, how many distinct values it takes, and whether any column is an identifier or
could leak the answer.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

## 3. Exact duplicates and the train/test split  *(rubric B1, B2)*

**What:** remove rows that are exact copies of an earlier row, then split the data **once** into an
80 % training set and a 20 % held-out test set that is used only for the final evaluation.

**Why the split comes this early:** every data-driven decision (EDA observations, imputation values,
scaling statistics, hyperparameters) must be made using the training data only. If we looked at the
test set first, our choices would be tuned to it and the reported test score would be optimistic.

**Why stratify a continuous target:** the rubric asks for a *stratified* split, which is normally
defined for class labels. Here the target is continuous, so it is cut into 10 quantile bins **only to
build the split** — each bin is represented in train and test in the same proportion. The bins are
never used as a feature and the target values are never changed (see `docs/clarifications.md`).

**Concept — held-out evaluation:** the test set estimates how the model performs on data it has
never seen. It is scored once, at the end.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

## 4. Exploratory data analysis on the training set  *(rubric A2, A3)*

**What:** distribution plots for every feature, the target distribution, a correlation heatmap and
scatter plots of features against the target.

**Why:** EDA shows the shape of each variable, which features are related to the target, and whether
features are related to each other (multicollinearity). This guides preprocessing and model choice.
EDA uses the **training set only**, so no information from the test set influences our decisions.

**Concept — correlation:** Pearson's *r* measures the strength of a *linear* relationship between two
numeric variables (−1 to +1). A low *r* does not rule out a non-linear relationship, which is why
scatter plots are shown as well.

Each plot is followed by a ✍️ team-observation cell.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

## 5. Missing values and outliers  *(rubric B1)*

**What:** count missing values per column, check for outliers with the inter-quartile-range (IQR)
rule and box plots, and decide how each is treated.

**Why:** most scikit-learn estimators cannot handle missing values, and extreme values can dominate
distance-based and squared-error models. Even when none are found, the check and the chosen strategy
must be documented.

**Concept — IQR rule:** a value is flagged as a potential outlier if it lies more than 1.5 × IQR below
the first quartile or above the third quartile. Flagged values are investigated, not deleted
automatically — an extreme value can be genuine.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

> ### ✍️ TEAM ANALYSIS REQUIRED — `REG-B1`: Cleaning decisions and their justification
> **Author:** _(Regression track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Were any missing values found? What imputation strategy is in the pipeline anyway, and why is it reasonable?
> - Exact duplicates were removed before the split, while rows that repeat only the feature values were kept. Why is that distinction defensible for this dataset?
> - Did the outlier check flag anything? What did you decide, and why?
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — REG-B1`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 6. Preprocessing: encoding and scaling inside a pipeline  *(rubric B2)*

**What:** one-hot encode the categorical column, standardise the numeric columns, and wrap these
steps together with each model in a scikit-learn `Pipeline`.

**Why:** a `Pipeline` guarantees that the encoder and scaler are **fitted on the training data only**
and merely *applied* to the test data — including inside every cross-validation fold. Fitting a scaler
on the full dataset before splitting leaks test-set statistics into training (guideline 7.1).

**Concept — standardisation:** `StandardScaler` rescales each feature to mean 0 and standard deviation
1 *using the training mean and standard deviation*. Distance-based (KNN, SVR) and regularised linear
models are sensitive to feature scale; tree-based models are not. All ten models share one
preprocessor so they are compared on the same preprocessed data.

This section includes a demonstration that the scaler really is fitted on the training set only.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

## 7. Feature engineering  *(rubric B3)*

**What:** add at least one engineered feature and measure whether it changes model performance.

**Why:** a well-chosen derived feature can expose a relationship that the original columns only
express indirectly.

**How it works here:** the team registers the feature in `src/feature_engineering.py` (section *TEAM
REGISTRATIONS*). The feature is added by a pipeline step, so it is computed identically for the
training and test data. The cell in this section then shows results with and without it. Until a
feature is registered, the pipeline runs unchanged and rubric item B3 is reported as incomplete.

> 🔧 **Build status:** the code for this section is added in **Phase 3**. This placeholder is replaced when that phase is built.

> ### ✍️ TEAM ANALYSIS REQUIRED — `REG-B3`: Engineered feature and justification
> **Author:** _(Regression track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Which feature did you create, and from which columns?
> - Why might it improve performance — what relationship does it capture that the raw columns do not?
> - Is it computed row by row (no statistics across rows), so that it cannot leak test information?
> - After measuring: did it actually help? Report the numbers, including if it did not.
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — REG-B3`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 8. The ten regression algorithms  *(rubric C1)*

**What:** train each of the ten algorithms required by guideline 3.1 on the same training set and
evaluate it on the same held-out test set.

**Why the same split:** comparisons are only fair if every model sees exactly the same training rows
and is scored on exactly the same test rows.

**How each subsection is organised:**
1. what the algorithm is, in plain words;
2. how it works, step by step;
3. the settings worth tuning;
4. a likely viva question and its answer;
5. code that builds the pipeline, trains it and records its test metrics;
6. the output specific to that algorithm (coefficients, sparsity, depth curve, importances …).

**Metrics** (all in the original units of the Performance Index):
* **R²** — share of the target's variance explained (1 = perfect, 0 = no better than predicting the mean).
* **RMSE** — root mean squared error; penalises large errors more heavily.
* **MAE** — mean absolute error; the average size of an error.

### 8.1 Linear Regression

*In one line:* fits a straight-line (hyperplane) relationship by ordinary least squares; the baseline, with interpretable coefficients.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.2 Ridge Regression

*In one line:* linear regression with an L2 penalty that shrinks coefficients; tuned through `alpha`.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.3 Lasso Regression

*In one line:* linear regression with an L1 penalty that can set coefficients exactly to zero (feature sparsity).

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.4 ElasticNet Regression

*In one line:* combines the L1 and L2 penalties; tuned through `alpha` and `l1_ratio`.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.5 Polynomial Regression

*In one line:* `PolynomialFeatures` followed by Linear Regression, adding squared and interaction terms; degrees 1, 2 and 3 are compared.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.6 Decision Tree Regressor

*In one line:* recursively splits the data on feature thresholds; tuned through `max_depth`, with feature importances.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.7 Random Forest Regressor

*In one line:* averages many decision trees trained on bootstrap samples; tuned through `n_estimators`.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.8 Gradient Boosting Regressor

*In one line:* adds shallow trees one at a time, each correcting the previous errors; tuned through `learning_rate`.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.9 Support Vector Regressor (SVR)

*In one line:* fits a function within an ε-tube around the data; requires scaled features; tuned through `C` and `kernel`.

> 🔧 **Build status:** implemented in **Phase 4**.

### 8.10 K-Nearest Neighbors Regressor

*In one line:* predicts the average target of the *k* closest training rows; tuned through `k`; scaling matters.

> 🔧 **Build status:** implemented in **Phase 4**.

## 9. Comparison of all ten models  *(rubric C2)*

**What:** a single table of R², RMSE and MAE for all ten models on the same test split, ranked by R².

**Why:** guideline 7.2 requires results in a summary table rather than scattered print statements.
The table is assembled from the results recorded in section 8, so it cannot disagree with them.

> 🔧 **Build status:** the code for this section is added in **Phase 5**. This placeholder is replaced when that phase is built.

## 10. 5-fold cross-validation of the leading models  *(guidelines 3.1 and 7.2)*

**What:** 5-fold cross-validated R² computed on the **training set**, used to nominate the two
leading models.

**Why:** a single test split gives one number that depends on which rows happened to land in the test
set. Cross-validation trains and scores the model five times on different training/validation
partitions, giving a mean and a spread. Doing it on the training set keeps the test set untouched.

**Concept — k-fold CV:** the training data is divided into k = 5 folds; each fold serves once as the
validation set while the model is trained on the other four. The preprocessing pipeline is refitted
inside every fold.

> 🔧 **Build status:** the code for this section is added in **Phase 5**. This placeholder is replaced when that phase is built.

## 11. Hyperparameter tuning  *(rubric C3)*

**What:** `GridSearchCV` on the two leading models, reporting the best parameters and the before/after
test metrics.

**Why:** default hyperparameters are rarely optimal. Grid search tries every combination in a grid and
selects the one with the best **cross-validated** score on the training data. The test set is used
only once, afterwards, to report the tuned model.

If tuning does not improve a model, the table says so.

> 🔧 **Build status:** the code for this section is added in **Phase 5**. This placeholder is replaced when that phase is built.

## 12. Diagnostics of the best model  *(rubric C4)*

**What:** a residual plot and a predicted-vs-actual plot for the best model, and a feature-importance
plot for a tree-based model.

**Why:** a single score hides *where* a model goes wrong. Residual plots reveal systematic patterns
(for example under-predicting high scores) that R² alone cannot show.

**Concept — residual:** actual − predicted. For a well-specified model, residuals scatter randomly
around zero with no trend.

> 🔧 **Build status:** the code for this section is added in **Phase 5**. This placeholder is replaced when that phase is built.

## 13. Saving the fitted pipelines

**What:** save each fitted pipeline (preprocessing + model) to `models/` with compressed `joblib`.

**Why:** a saved pipeline can make predictions on new raw data without retraining and without
re-implementing preprocessing. File sizes are checked so large ensembles are not committed by accident.

> 🔧 **Build status:** the code for this section is added in **Phase 5**. This placeholder is replaced when that phase is built.

## 14. Model selection and conclusions

> ### ✍️ TEAM ANALYSIS REQUIRED — `REG-CONCLUSION`: Model selection, conclusions and limitations
> **Author:** _(Regression track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Which model would you select, and why? Consider test metrics, cross-validation mean and spread, and simplicity.
> - Why do you think the models ranked the way they did? Relate this to what the EDA showed about the data.
> - What did hyperparameter tuning change, and was the improvement meaningful?
> - What limits these conclusions (for example the synthetic nature of the data)?
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — REG-CONCLUSION`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 15. References and AI-assistance disclosure

* 23CSE301 Machine Learning — Capstone Project Guidelines, Algorithm List & Rubrics (2026-27).
* Dataset: N. Narayan, *Student Performance (Multiple Linear Regression)*, Kaggle —
  https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression
  (see `docs/dataset_sources.md` for checksum and licence notes).
* scikit-learn documentation — https://scikit-learn.org/stable/

**AI assistance (guideline 7.5):** an AI coding assistant generated the code scaffolding, the
supporting `src/` modules and the factual algorithm explanations. It did **not** write the EDA
observations, the feature-engineering justification, the model-selection reasoning or the conclusions
— those are the ✍️ team cells. Full disclosure: `README.md`.